In [15]:
import joblib

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from loguru import logger

%matplotlib inline

pd.set_option("display.max_columns", None)

In [16]:
from src.evaluation import build_predictions_report

In [17]:
def plot_barh_by_model(results_df, level_label, x, y="model", hue=None, title="", figsize=(8, 6)):
    """Barplot horizontal genérico para comparar modelos por una métrica."""
    palette = {"ML": "#1f77b4", "Statistical": "#ff7f0e", "Naive": "#2ca02c"}
    df_sorted = results_df.sort_values(x, ascending=True)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(df_sorted, x=x, y=y, hue=hue, hue_order=list(palette.keys()), order=df_sorted[y],
                dodge=False, palette=palette if hue else None, ax=ax)
    ax.set_title(f"{level_label} | {title}", loc="left", pad=20)
    ax.set_xlabel("")
    ax.set_ylabel("")
    sns.despine()
    if hue:
        ax.legend(title=hue, loc="upper right")
    plt.tight_layout()
    plt.show()

In [18]:
!ls ../artifacts/models/

level_01_daily_total_sales_artifact.pkl
level_01_weekly_total_sales_artifact.pkl
level_02_daily_state_sales_artifact.pkl
level_02_weekly_state_sales_artifact.pkl
level_03_daily_cat_sales_artifact.pkl
level_03_weekly_cat_sales_artifact.pkl
level_04_daily_dept_sales_artifact.pkl
level_04_weekly_dept_sales_artifact.pkl
level_05_daily_state_cat_sales_artifact.pkl
level_05_weekly_state_cat_sales_artifact.pkl
level_06_daily_store_sales_artifact.pkl
level_06_weekly_store_sales_artifact.pkl
level_07_daily_state_dept_sales_artifact.pkl
level_07_weekly_state_dept_sales_artifact.pkl
level_08_daily_store_cat_sales_artifact.pkl
level_08_weekly_store_cat_sales_artifact.pkl
level_09_daily_store_dept_sales_artifact.pkl
level_09_weekly_store_dept_sales_artifact.pkl
registry.json
versions


In [19]:
ARTIFACT = "level_03_daily_cat_sales_artifact"

In [20]:
import joblib
from src.data.dataset import reconstruct_test_data

artifact_path = f"../artifacts/models/{ARTIFACT}.pkl"
artifact = joblib.load(artifact_path)

grainly = "daily" if "daily" in artifact_path else "weekly"
level_label = artifact["level_label"]
model = artifact["model"]
FEATURES = artifact["features"]
feature_importance = artifact["feature_importance"]
results_df = artifact["results_df"]

X_test, y_test, artifact["train"], artifact["test"] = reconstruct_test_data(artifact)

2026-09-17 00:13:08.361 | INFO     | src.data.dataset:load_data:58 - level_03_daily_cat | target=sales | 5,823 filas | 191 features (9 categóricas)
2026-09-17 00:13:08.427 | INFO     | src.data.temporal_split:log_summary:41 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 4,647 rows)
2026-09-17 00:13:08.430 | INFO     | src.data.temporal_split:log_summary:42 - Valid : 2015-04-27 to 2016-04-24 (364 days, 1,092 rows)
2026-09-17 00:13:08.434 | INFO     | src.data.temporal_split:log_summary:43 - Test  : 2016-04-25 to 2016-05-22 (28 days, 84 rows)
2026-09-17 00:13:12.607 | INFO     | src.data.dataset:split_data:119 - Features: 57 (2 categóricas)


In [31]:
import dataframe_image as dfi

df_sample = X_test.head(8).T

dfi.export(df_sample, 'df_sample.png', table_conversion='matplotlib',max_cols=8, max_rows=25, fontsize=12)
df_sample

,5739,5740,5741,5742,5743,5744,5745,5746
cat_id,HOBBIES,HOUSEHOLD,FOODS,FOODS,HOBBIES,HOUSEHOLD,FOODS,HOUSEHOLD
event_name_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_change,0.015571,0.003837,-0.014164,-0.00808,0.008515,0.008495,-0.024084,-0.007978
price_vs_mean,1.200342,1.060269,1.107955,1.112379,1.147018,1.056603,1.100907,1.034374
day,25,25,25,26,26,26,27,27
weekofyear,17,17,17,17,17,17,17,17
dow_sin,0.0,0.0,0.0,0.781832,0.781832,0.781832,0.974928,0.974928
month_sin,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
month_cos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
price_volatility,0.195193,0.093317,0.051669,0.051685,0.195851,0.09417,0.051595,0.093651


In [ ]:
results_df = results_df.query("model != 'xgboost (final, bench default)'")

In [ ]:
for col in ['wape', 'wrmsse']:
    dict_rename = {
        'wape': 'WAPE',
        'wrmsse': 'WRMSSE'
    }
    plot_barh_by_model(results_df, level_label, hue="category", x=col, y="model", title=f"{dict_rename[col]} por modelo")

## Explicación modelo

In [ ]:
import shap

explainer = artifact["explainer"]

X_shap = model.prepare(X_test.sample(n=min(300_000, len(X_test)), random_state=42))

cat_cols = [c for c in X_shap.columns if isinstance(X_shap[c].dtype, pd.CategoricalDtype)]
if cat_cols:
    X_shap = X_shap.assign(**{c: X_shap[c].cat.codes for c in cat_cols})

shap_values = explainer.shap_values(X_shap)
shap_df = pd.DataFrame(shap_values, columns=X_shap.columns, index=X_shap.index)

importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

In [ ]:
top = 10

shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    max_display=top,
    show=False,
    plot_size=(8, 4),
    color="#1f77b4",
)

plt.title(f"{level_label}: Importancia de variables (SHAP) | Top {top}", fontsize=14)
plt.xlabel("Impacto promedio en la predicción", fontsize=12)
plt.show()

# Predicciones

In [ ]:
from src.evaluation import analizar_prediccion, plot_forecast

In [ ]:
metrics_test_final, df_pred = build_predictions_report(artifact['train'], artifact['test'], y_test, model.predict(X_test), target_col=artifact['target'])

print(f"Total sales: {df_pred['sales'].sum():,.0f}")
print(f"Total predicted: {df_pred['y_pred'].sum():,.0f}")
print(f"Diff: {df_pred['y_pred'].sum() - df_pred['sales'].sum():,.0f}")

print(f"Test WAPE: {metrics_test_final['wape']:.1%}")
print(f"Test BIAS: {metrics_test_final['bias']:.1%}")

print(f"Test WRMSSE: {metrics_test_final['wrmsse']:.3f}")

In [ ]:
from src.evaluation import build_series_metrics

df_metrics = build_series_metrics(artifact['train'], df_pred, target_col=artifact['target'], weight_level=["date"])
df_metrics.head(5)

In [ ]:
df_metrics.sort_values('wape').head()

In [ ]:
for i in df_metrics.sort_values('gross_sales_pct', ascending=False).head(3).index:
    plot_forecast(df_pred, level_label, 'sales', series_id=i)

In [ ]:
fecha = "2016-05-07"

test_meta = artifact["test"]
row_idx = test_meta.index[test_meta["date"] == pd.Timestamp(fecha)]

if row_idx.empty:
    raise ValueError(f"No hay datos de test para la fecha {fecha}")
if row_idx[0] not in X_shap.index:
    raise ValueError(f"La fecha {fecha} no fue muestreada en X_shap")

idx = X_shap.index.get_loc(row_idx[0])

base_value = np.ravel(explainer.expected_value)[0]

plt.figure(figsize=(8, 5))

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx],
        base_values=base_value,
        data=X_shap.iloc[idx],
        feature_names=X_shap.columns.tolist()
    ),
    max_display=15,
    show=False
)

fig = plt.gcf()

# recorre TODOS los ejes de la figura, no solo el actual
for ax in fig.axes:
    for text in ax.texts:
        text.set_visible(False)
        
plt.title(f"{level_label}: Explicación SHAP para {fecha}", fontsize=14, pad=60)
plt.tight_layout()
plt.show()

# Sandox

In [ ]:
SERIES = 'HOUSEHOLD_CA_1_CA'
df_sample = df_pred.query(f'series_id == "{SERIES}"').copy()

#display(df_sample)

top_best_dates = df_pred.query(f'series_id == "{SERIES}"').nsmallest(2, 'wape')['date']
top_error_dates = df_pred.query(f'series_id == "{SERIES}"').nlargest(2, 'wape')['date']

#plot_forecast(df_pred, 'sales', series_id=SERIES)
#plot_forecast(df_pred, 'gross_sales', series_id=SERIES)
#plot_forecast(df_pred, 'wape', series_id=SERIES)
#plot_forecast(df_pred, 'bias', series_id=SERIES)

In [ ]:
SERIES = "FOODS_2_FOODS"

for i in top_error_dates:
    print("#"*50,f"Date: {i}", "#"*50)
    analizar_prediccion(test_df=artifact['test'], X_test=X_test, df_pred=df_pred, target_col=artifact['target'], series_id=SERIES, explainer=explainer, date=i)

In [ ]:
for i in top_best_dates:
    print("#"*50,f"Date: {i}", "#"*50)
    analizar_prediccion(test_df=artifact['test'], X_test=X_test, df_pred=df_pred, target_col=artifact['target'], series_id=SERIES, explainer=explainer, date=i)